In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

### Data Preprocessing

In [2]:
df = pd.read_csv("./extracted_features.csv")

In [3]:
df.head()

,file_name,class_name,start_sec,end_sec,chroma_0_mean,chroma_0_stdev,chroma_1_mean,chroma_1_stdev,chroma_2_mean,chroma_2_stdev,...,mfcc_30_mean,mfcc_30_stdev,mfcc_31_mean,mfcc_31_stdev,mfcc_combined_mean,mfcc_combined_stdev,sc_mean,sc_stdev,zcr_mean,zcr_stdev
0,-0CamVQdP_Y.wav,music,0,1,0.066293,0.176518,0.048511,0.172104,0.051913,0.162008,...,1.111148,11.747592,21.096506,11.658666,-8.634419,45.127710,1804.285910,647.639159,0.068659,0.035247
1,-0CamVQdP_Y.wav,music,1,2,0.040602,0.028222,0.005886,0.005564,0.013600,0.010802,...,-16.738794,4.011196,21.843700,6.849890,-7.618063,41.428406,1813.984224,225.661075,0.076760,0.019730
2,-0CamVQdP_Y.wav,music,2,3,0.141798,0.115657,0.149388,0.123707,0.090743,0.069804,...,-4.915877,8.996728,11.760752,8.297294,-5.247653,28.779720,2674.215001,498.322674,0.126365,0.032523
3,-0CamVQdP_Y.wav,music,3,4,0.260232,0.256056,0.274225,0.171083,0.450356,0.350174,...,0.233849,4.196388,0.825368,6.892590,-4.867526,25.096298,2919.190251,363.122534,0.137029,0.031587
4,-0CamVQdP_Y.wav,music,4,5,0.234812,0.187384,0.304931,0.170424,0.588712,0.382219,...,0.266489,4.053829,-0.212216,4.406050,-3.923183,20.140772,3431.760810,975.948634,0.190652,0.094494


In [4]:
# drop unused columns for modelling

df = df.drop(["file_name", "start_sec", "end_sec"], axis="columns")

In [5]:
# class name ordinal encoding

class_mappings = {
    "music": 0,
    "speech": 1,
    "noise": 2
}

df["class_name"] = df["class_name"].map(class_mappings)

In [6]:
# train test split

X = df.drop("class_name", axis="columns")
y = df["class_name"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

X_test.head()

,chroma_0_mean,chroma_0_stdev,chroma_1_mean,chroma_1_stdev,chroma_2_mean,chroma_2_stdev,chroma_3_mean,chroma_3_stdev,chroma_4_mean,chroma_4_stdev,...,mfcc_30_mean,mfcc_30_stdev,mfcc_31_mean,mfcc_31_stdev,mfcc_combined_mean,mfcc_combined_stdev,sc_mean,sc_stdev,zcr_mean,zcr_stdev
7185,0.159440,0.228138,0.145875,0.222041,0.221493,0.273171,0.294411,0.291727,0.326161,0.347100,...,-4.260369,12.106551,-0.203332,14.684827,-4.208765,45.558502,1965.380928,325.818531,0.122048,0.039712
4880,0.607098,0.199338,0.603993,0.161502,0.692209,0.202975,0.618437,0.154984,0.558640,0.152697,...,-5.196217,3.387196,3.088835,3.138841,-3.468067,22.477710,2748.310481,251.708911,0.185103,0.036124
5215,0.388995,0.288331,0.292953,0.251116,0.348644,0.291691,0.239947,0.238587,0.149403,0.124596,...,-4.399375,5.844445,-4.404249,5.114483,-8.692061,27.271687,1920.314673,163.059400,0.107644,0.017030
466,0.043712,0.031931,0.044944,0.031817,0.076663,0.064562,0.273333,0.238871,0.443032,0.298910,...,-11.908316,3.502922,8.025697,5.997826,-0.597672,30.385975,2309.143148,249.463066,0.126776,0.038780
10087,0.758179,0.279207,0.609906,0.321393,0.243221,0.186654,0.406935,0.262261,0.129224,0.069946,...,3.294733,5.038076,6.007130,4.658214,-8.521165,22.296060,2298.684546,219.057991,0.145630,0.023871


### Model Training

In [7]:
# train model

xgboost = XGBClassifier()

xgboost.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [8]:
# get prediction scores

y_train_pred = xgboost.predict(X_train)
y_test_pred = xgboost.predict(X_test)

print(f"train accuracy: {accuracy_score(y_train_pred, y_train)}")
print(f"test accuracy: {accuracy_score(y_test_pred, y_test)}")

train accuracy: 0.9083130609848167
test accuracy: 0.6528301886792452


In [9]:
# save model

xgboost.save_model("model_xgboost.json")